In [1]:
import sklearn
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.model_selection import train_test_split
import torch

In [2]:
N = 100000
L = 200
beta = 0.3
sys_data = np.load(f"../data/Fast_CritIsing_L{L}_beta{beta}_Zoutcomes_sys.npy")
anc_data = np.load(f"../data/Fast_CritIsing_L{L}_beta{beta}_Zoutcomes_anc.npy")
X = anc_data[:N]
y = sys_data[:N]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# TODO: I want to train a dilated CNN to predict the output y (a list of {+1, -1} of length L=200, representing the Z measurement outcome on the system) from the input X (a list of {+1, -1} of length L, representing the Z measurement outcome on the ancilla). The size of the dataset is N=100000, and I have already split it into training and validation set. The system has Fixed Boundary Conditions, meaning that the first and the last entry of each row of X and y are fixed to +1. We can define the loss function as the mean squared error between the predicted output and the true output, or we can also use the binary cross-entropy loss if we treat the problem as a classification problem. Remember that for a regression model, a physically meaningful output should be between -1 and 1. We need to print both the training and validation loss, and also print the predicted output for the uniform input (all +1). Remember to set the random seed for reproducibility. Remember to print the total number of parameters in the model.

In [11]:
# Dilated CNN for predicting system Z outcomes from ancilla Z outcomes
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_t = torch.from_numpy(y_train.astype(np.float32))
y_valid_t = torch.from_numpy(y_valid.astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t, y_train_t, y_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_t)
valid_ds = TensorDataset(X_valid_t, y_valid_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class DilatedCNN(nn.Module):
    def __init__(self, hidden_channels=64, depth=4, kernel_size=3):
        super().__init__()
        layers = []
        in_ch = 1
        for i in range(depth):
            dilation = 2 ** i
            padding = dilation  # keeps length with kernel_size=3
            layers.append(nn.Conv1d(in_ch, hidden_channels, kernel_size, padding=padding, dilation=dilation))
            layers.append(nn.ReLU())
            in_ch = hidden_channels
        layers.append(nn.Conv1d(in_ch, 1, kernel_size=3, padding=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.unsqueeze(1)
        out = self.net(x)
        out = torch.tanh(out)
        return out.squeeze(1)

model = DilatedCNN().to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count}")
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).squeeze(0).cpu().numpy()
print("Predicted output for all +1 input:")
print(uniform_pred)


Model parameters: 37505
Epoch 1/10 - train_loss: 0.4779 - val_loss: 0.4611
Epoch 2/10 - train_loss: 0.4594 - val_loss: 0.4618
Epoch 3/10 - train_loss: 0.4589 - val_loss: 0.4598
Epoch 4/10 - train_loss: 0.4588 - val_loss: 0.4595
Epoch 5/10 - train_loss: 0.4587 - val_loss: 0.4608
Epoch 6/10 - train_loss: 0.4581 - val_loss: 0.4597
Epoch 7/10 - train_loss: 0.4584 - val_loss: 0.4597
Epoch 8/10 - train_loss: 0.4582 - val_loss: 0.4601
Epoch 9/10 - train_loss: 0.4581 - val_loss: 0.4612
Epoch 10/10 - train_loss: 0.4581 - val_loss: 0.4591
Predicted output for all +1 input:
[0.9489385  0.91776973 0.92318135 0.93096536 0.93742263 0.95803726
 0.960353   0.949558   0.9433772  0.9538616  0.95393866 0.954407
 0.95435876 0.9555663  0.9556067  0.9569636  0.95771694 0.95771694
 0.95771694 0.95771694 0.95771694 0.95771694 0.95771694 0.95771694
 0.95771694 0.95771694 0.95771694 0.95771694 0.95771694 0.95771694
 0.95771694 0.95771694 0.95771694 0.95771694 0.95771694 0.95771694
 0.95771694 0.95771694 0.95771

In [ ]:
# Dilated CNN to predict a single site r (y[:, r]) from X
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

r = 20  # site index to predict (0-based); change as needed

seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_site = torch.from_numpy(y_train[:, r].astype(np.float32))
y_valid_site = torch.from_numpy(y_valid[:, r].astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_site)
valid_ds = TensorDataset(X_valid_t, y_valid_site)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class DilatedCNNOneSite(nn.Module):
    def __init__(self, hidden_channels=64, depth=4, kernel_size=3, site_index=0):
        super().__init__()
        self.site_index = site_index
        layers = []
        in_ch = 1
        for i in range(depth):
            dilation = 2 ** i
            padding = dilation  # keep length with kernel_size=3
            layers.append(nn.Conv1d(in_ch, hidden_channels, kernel_size, padding=padding, dilation=dilation))
            layers.append(nn.ReLU())
            in_ch = hidden_channels
        layers.append(nn.Conv1d(in_ch, 1, kernel_size=3, padding=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.unsqueeze(1)
        out = self.net(x)
        out = torch.tanh(out).squeeze(1)
        return out[:, self.site_index]

model = DilatedCNNOneSite(site_index=r).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).item()
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")


Model parameters: 37505
Epoch 1/5 - train_loss: 0.4937 - val_loss: 0.4727
Epoch 2/5 - train_loss: 0.4706 - val_loss: 0.4696
Epoch 3/5 - train_loss: 0.4679 - val_loss: 0.4694
Epoch 4/5 - train_loss: 0.4670 - val_loss: 0.4725
Epoch 5/5 - train_loss: 0.4663 - val_loss: 0.4708
Predicted output at site 10 for all +1 input: 0.9816486835479736


In [13]:
# Locally-connected (non-shared) dilated network for site r
import random
import math
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

r = 10  # site index to predict (0-based)
seed = 2
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_site = torch.from_numpy(y_train[:, r].astype(np.float32))
y_valid_site = torch.from_numpy(y_valid[:, r].astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_site)
valid_ds = TensorDataset(X_valid_t, y_valid_site)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class LocallyConnected1d(nn.Module):
    def __init__(self, length, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        self.length = length
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.dilation = dilation
        in_features = in_channels * kernel_size
        self.weight = nn.Parameter(torch.empty(length, out_channels, in_features))
        self.bias = nn.Parameter(torch.zeros(length, out_channels))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = in_features
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):  # x: (B, C, L)
        B, C, L = x.shape
        assert L == self.length, "Input length mismatch"
        pad = self.dilation
        x_pad = F.pad(x, (pad, pad), value=1.0)  # preserve fixed boundaries
        patches = F.unfold(x_pad.unsqueeze(2), kernel_size=(1, self.kernel_size),
                          dilation=(1, self.dilation), padding=(0, 0), stride=(1, 1))
        patches = patches.view(B, C * self.kernel_size, L)  # (B, Cin*k, L)
        patches = patches.permute(0, 2, 1)  # (B, L, Cin*k)
        out = torch.einsum('bli,loj->blo', patches, self.weight) + self.bias  # (B, L, Cout)
        return out.permute(0, 2, 1)  # (B, Cout, L)

class LocalDilatedCNNOneSite(nn.Module):
    def __init__(self, length=L, hidden_channels=32, depth=3, kernel_size=3, site_index=0):
        super().__init__()
        layers = []
        in_ch = 1
        for i in range(depth):
            dilation = 2 ** i
            layers.append(LocallyConnected1d(length, in_ch, hidden_channels, kernel_size, dilation))
            layers.append(nn.ReLU())
            in_ch = hidden_channels
        layers.append(LocallyConnected1d(length, in_ch, 1, kernel_size, dilation=1))
        self.net = nn.Sequential(*layers)
        self.site_index = site_index

    def forward(self, x):
        x = x.unsqueeze(1)  # (B,1,L)
        out = self.net(x)   # (B,1,L)
        out = torch.tanh(out)  # keep outputs in [-1,1]
        return out[:, 0, self.site_index]

model = LocalDilatedCNNOneSite(site_index=r).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters (non-shared): {param_count}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).item()
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")


Model parameters (non-shared): 1286600
Epoch 1/5 - train_loss: 2.0009 - val_loss: 1.9902
Epoch 2/5 - train_loss: 2.0009 - val_loss: 1.9902
Epoch 3/5 - train_loss: 2.0009 - val_loss: 1.9902
Epoch 4/5 - train_loss: 2.0009 - val_loss: 1.9902
Epoch 5/5 - train_loss: 2.0009 - val_loss: 1.9902
Predicted output at site 10 for all +1 input: -1.0


In [14]:
# Dilated CNN with positional channel + per-site linear head (site r)
import random
import math
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

r = 10  # site index to predict (0-based)
seed = 3
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_site = torch.from_numpy(y_train[:, r].astype(np.float32))
y_valid_site = torch.from_numpy(y_valid[:, r].astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_site)
valid_ds = TensorDataset(X_valid_t, y_valid_site)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class DilatedPosCNNOneSite(nn.Module):
    def __init__(self, length=L, hidden_channels=64, depth=4, kernel_size=3, site_index=0):
        super().__init__()
        layers = []
        in_ch = 2  # measurement + position channel
        for i in range(depth):
            dilation = 2 ** i
            padding = dilation  # keep length for kernel_size=3
            layers.append(nn.Conv1d(in_ch, hidden_channels, kernel_size, padding=padding, dilation=dilation))
            layers.append(nn.ReLU())
            in_ch = hidden_channels
        self.backbone = nn.Sequential(*layers)
        self.register_buffer("pos_channel", torch.linspace(0.0, 1.0, length).unsqueeze(0))
        self.head_weight = nn.Parameter(torch.empty(length, hidden_channels))
        self.head_bias = nn.Parameter(torch.zeros(length))
        nn.init.kaiming_uniform_(self.head_weight, a=math.sqrt(5))
        fan_in = hidden_channels
        bound = 1 / math.sqrt(fan_in)
        nn.init.uniform_(self.head_bias, -bound, bound)
        self.site_index = site_index

    def forward(self, x):
        pos = self.pos_channel.expand(x.size(0), -1)
        x = torch.stack((x, pos), dim=1)  # (B,2,L)
        feat = self.backbone(x)  # (B, hidden, L)
        feat_site = feat[:, :, self.site_index]
        w = self.head_weight[self.site_index]
        b = self.head_bias[self.site_index]
        out = (feat_site * w).sum(dim=1) + b
        return torch.tanh(out)

model = DilatedPosCNNOneSite(site_index=r).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters (pos-aware per-site head): {param_count}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).item()
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")


Model parameters (pos-aware per-site head): 50504
Epoch 1/5 - train_loss: 0.5018 - val_loss: 0.4759
Epoch 2/5 - train_loss: 0.4714 - val_loss: 0.4801
Epoch 3/5 - train_loss: 0.4691 - val_loss: 0.4709
Epoch 4/5 - train_loss: 0.4668 - val_loss: 0.4695
Epoch 5/5 - train_loss: 0.4669 - val_loss: 0.4696
Predicted output at site 10 for all +1 input: 0.9765872359275818


In [4]:
# Segment-wise (axial) attention model for site r
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

r = 10  # site index to predict (0-based)
seed = 4
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_site = torch.from_numpy(y_train[:, r].astype(np.float32))
y_valid_site = torch.from_numpy(y_valid[:, r].astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_site)
valid_ds = TensorDataset(X_valid_t, y_valid_site)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class SegmentAttentionOneSite(nn.Module):
    def __init__(self, length=L, d_model=64, nhead=4, seg_len=25, site_index=0):
        super().__init__()
        assert length % seg_len == 0, "seg_len must divide length"
        self.length = length
        self.seg_len = seg_len
        self.num_seg = length // seg_len
        self.site_index = site_index

        self.embed = nn.Linear(2, d_model)  # measurement + position
        self.attn_in_seg = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.attn_between_seg = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, d_model))
        self.head = nn.Linear(d_model, 1)
        self.register_buffer("pos", torch.linspace(0.0, 1.0, length).unsqueeze(0))

    def forward(self, x):
        # x: (B, L)
        pos = self.pos.expand(x.size(0), -1)
        inp = torch.stack((x, pos), dim=-1)  # (B, L, 2)
        feat = self.embed(inp)  # (B, L, d)

        feat = feat.view(x.size(0), self.num_seg, self.seg_len, -1)
        seg_flat = feat.view(-1, self.seg_len, feat.size(-1))
        seg_out, _ = self.attn_in_seg(seg_flat, seg_flat, seg_flat)
        seg_out = seg_out + seg_flat  # residual
        seg_out = seg_out.view(x.size(0), self.num_seg, self.seg_len, -1)

        axial_in = seg_out.permute(0, 2, 1, 3).contiguous()  # (B, seg_len, num_seg, d)
        axial_flat = axial_in.view(-1, self.num_seg, seg_out.size(-1))
        axial_out, _ = self.attn_between_seg(axial_flat, axial_flat, axial_flat)
        axial_out = axial_out + axial_flat
        axial_out = axial_out.view(x.size(0), self.seg_len, self.num_seg, -1).permute(0, 2, 1, 3)

        out = axial_out.reshape(x.size(0), self.length, -1)
        out = self.ff(out) + out

        site_feat = out[:, self.site_index, :]
        pred = torch.tanh(self.head(site_feat).squeeze(-1))
        return pred

model = SegmentAttentionOneSite(site_index=r).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters (segment attention, site {r}): {param_count}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).item()
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")


Model parameters (segment attention, site 10): 41857
Epoch 1/5 - train_loss: 0.5082 - val_loss: 0.4982
Epoch 2/5 - train_loss: 0.4905 - val_loss: 0.4907
Epoch 3/5 - train_loss: 0.4898 - val_loss: 0.4957
Epoch 4/5 - train_loss: 0.4891 - val_loss: 0.4909
Epoch 5/5 - train_loss: 0.4884 - val_loss: 0.4909
Predicted output at site 10 for all +1 input: 0.9733083844184875


In [5]:
# U-Net encoder-decoder with skips (site r)
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

r = 10  # site index to predict (0-based)
seed = 5
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_site = torch.from_numpy(y_train[:, r].astype(np.float32))
y_valid_site = torch.from_numpy(y_valid[:, r].astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_site)
valid_ds = TensorDataset(X_valid_t, y_valid_site)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class UNet1DOneSite(nn.Module):
    def __init__(self, length=L, base_channels=32, site_index=0):
        super().__init__()
        self.site_index = site_index
        self.register_buffer("pos", torch.linspace(0.0, 1.0, length).unsqueeze(0))

        self.enc1 = nn.Sequential(
            nn.Conv1d(2, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.enc2 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.enc3 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.bottleneck = nn.Sequential(
            nn.Conv1d(base_channels * 4, base_channels * 4, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.up2 = nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.Sequential(
            nn.Conv1d(base_channels * 4, base_channels * 2, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.up1 = nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec1 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.out_conv = nn.Conv1d(base_channels, 1, kernel_size=1)

    def forward(self, x):
        pos = self.pos.expand(x.size(0), -1)
        x = torch.stack((x, pos), dim=1)  # (B,2,L)

        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b = self.bottleneck(e3)

        u2 = self.up2(b)
        u2 = torch.cat([u2, e2], dim=1)
        d2 = self.dec2(u2)

        u1 = self.up1(d2)
        u1 = torch.cat([u1, e1], dim=1)
        d1 = self.dec1(u1)

        out = self.out_conv(d1)  # (B,1,L)
        return torch.tanh(out[:, 0, self.site_index])

model = UNet1DOneSite(site_index=r).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters (1D U-Net, site {r}): {param_count}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).item()
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")


Model parameters (1D U-Net, site 10): 152321
Epoch 1/5 - train_loss: 0.4869 - val_loss: 0.4734
Epoch 2/5 - train_loss: 0.4715 - val_loss: 0.4674
Epoch 3/5 - train_loss: 0.4677 - val_loss: 0.4679
Epoch 4/5 - train_loss: 0.4678 - val_loss: 0.4714
Epoch 5/5 - train_loss: 0.4674 - val_loss: 0.4709
Predicted output at site 10 for all +1 input: 0.9741203188896179


In [7]:
# Chebyshev graph convolution on 1D chain (site r)
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import math

r = 10  # site index to predict (0-based)
seed = 6
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.from_numpy(X_train.astype(np.float32))
X_valid_t = torch.from_numpy(X_valid.astype(np.float32))
y_train_site = torch.from_numpy(y_train[:, r].astype(np.float32))
y_valid_site = torch.from_numpy(y_valid[:, r].astype(np.float32))
with torch.no_grad():
    for t in (X_train_t, X_valid_t):
        t[:, [0, -1]] = 1.0

train_ds = TensorDataset(X_train_t, y_train_site)
valid_ds = TensorDataset(X_valid_t, y_valid_site)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)

class ChebConv1D(nn.Module):
    def __init__(self, length, in_channels, out_channels, K=3):
        super().__init__()
        self.length = length
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.K = K
        self.weight = nn.Parameter(torch.empty(K * in_channels, out_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        bound = 1 / math.sqrt(K * in_channels)
        nn.init.uniform_(self.bias, -bound, bound)
        # build normalized Laplacian for 1D chain with fixed boundaries
        L = torch.zeros(length, length)
        for i in range(length):
            if i > 0:
                L[i, i - 1] = -1.0
            if i < length - 1:
                L[i, i + 1] = -1.0
            deg = 2.0 if 0 < i < length - 1 else 1.0
            L[i, i] = deg
        # normalized Laplacian
        D_inv_sqrt = torch.diag(1.0 / torch.sqrt(torch.diag(L)))
        L_norm = torch.eye(length) - D_inv_sqrt @ (-L + torch.diag(torch.diag(L))) @ D_inv_sqrt
        L_tilde = L_norm - torch.eye(length)  # lambda_max ~ 2
        self.register_buffer('L_tilde', L_tilde)

    def forward(self, x):  # x: (B, N, C)
        B, N, C = x.shape
        assert N == self.length, 'Length mismatch'
        Tx_0 = x
        if self.K == 1:
            supports = [Tx_0]
        else:
            Tx_1 = torch.einsum('nm,bmc->bnc', self.L_tilde, x)
            supports = [Tx_0, Tx_1]
            for _ in range(2, self.K):
                Tx_2 = 2 * torch.einsum('nm,bmc->bnc', self.L_tilde, supports[-1]) - supports[-2]
                supports.append(Tx_2)
        out = torch.cat(supports, dim=2)  # (B, N, K*C)
        out = torch.einsum('bnc,cf->bnf', out, self.weight) + self.bias
        return out

class ChebGraphNetOneSite(nn.Module):
    def __init__(self, length=L, hidden=64, K=3, site_index=0):
        super().__init__()
        self.site_index = site_index
        self.register_buffer('pos', torch.linspace(0.0, 1.0, length).unsqueeze(0))
        self.embed = nn.Linear(2, hidden)
        self.cheb1 = ChebConv1D(length, hidden, hidden, K)
        self.act1 = nn.ReLU()
        self.cheb2 = ChebConv1D(length, hidden, 1, K)

    def forward(self, x):
        pos = self.pos.expand(x.size(0), -1)
        x = torch.stack((x, pos), dim=-1)  # (B, L, 2)
        x = self.embed(x)
        x = self.cheb1(x)
        x = self.act1(x)
        x = self.cheb2(x)
        out = torch.tanh(x[:, self.site_index, 0])
        return out

model = ChebGraphNetOneSite(site_index=r).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters (Chebyshev GCN, site {r}): {param_count}")

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(valid_loader.dataset)
    print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

model.eval()
with torch.no_grad():
    uniform_input = torch.ones(1, L, device=device)
    uniform_pred = model(uniform_input).item()
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")


Model parameters (Chebyshev GCN, site 10): 12737
Epoch 1/5 - train_loss: 0.5559 - val_loss: 0.5733
Epoch 2/5 - train_loss: 0.5314 - val_loss: 0.5289
Epoch 3/5 - train_loss: 0.5217 - val_loss: 0.5398
Epoch 4/5 - train_loss: 0.5215 - val_loss: 0.5149
Epoch 5/5 - train_loss: 0.5168 - val_loss: 0.5103
Predicted output at site 10 for all +1 input: 0.9920353889465332


In [8]:
# Linear regression baseline for site r
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

r = 10  # site index to predict (0-based)
np.random.seed(7)

X_train_lr = X_train.copy()
X_valid_lr = X_valid.copy()
# enforce fixed boundaries (already fixed in data, but keep explicit)
X_train_lr[:, [0, -1]] = 1
X_valid_lr[:, [0, -1]] = 1

y_train_lr = y_train[:, r]
y_valid_lr = y_valid[:, r]

linreg = LinearRegression()
linreg.fit(X_train_lr, y_train_lr)

train_pred = linreg.predict(X_train_lr)
valid_pred = linreg.predict(X_valid_lr)
train_mse = mean_squared_error(y_train_lr, train_pred)
valid_mse = mean_squared_error(y_valid_lr, valid_pred)
print(f"Linear regression (site {r}) train MSE: {train_mse:.4f} - val MSE: {valid_mse:.4f}")

uniform_input = np.ones((1, L))
uniform_pred = linreg.predict(uniform_input)[0]
print(f"Predicted output at site {r} for all +1 input: {uniform_pred}")

param_count = linreg.coef_.size + 1  # weights + bias
print(f"Number of parameters: {param_count}")


Linear regression (site 10) train MSE: 0.4785 - val MSE: 0.4823
Predicted output at site 10 for all +1 input: 1.7841633068009877
Number of parameters: 201


In [19]:
# Logistic regression baseline (site r) with expectation-valued MSE
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, mean_squared_error

r = 100  # site index to predict (0-based)
np.random.seed(8)

X_train_lr = X_train.copy()
X_valid_lr = X_valid.copy()
# enforce fixed boundaries explicitly
X_train_lr[:, [0, -1]] = 1
X_valid_lr[:, [0, -1]] = 1

y_train_bin = ((y_train[:, r] + 1) / 2).astype(int)
y_valid_bin = ((y_valid[:, r] + 1) / 2).astype(int)

logreg = LogisticRegression(max_iter=200, solver='lbfgs')
logreg.fit(X_train_lr, y_train_bin)

train_proba = logreg.predict_proba(X_train_lr)[:, 1]
valid_proba = logreg.predict_proba(X_valid_lr)[:, 1]

train_logloss = log_loss(y_train_bin, train_proba)
valid_logloss = log_loss(y_valid_bin, valid_proba)
print(f"Logistic regression (site {r}) train logloss: {train_logloss:.4f} - val logloss: {valid_logloss:.4f}")

train_exp = 2 * train_proba - 1
valid_exp = 2 * valid_proba - 1
train_mse = mean_squared_error(y_train[:, r], train_exp)
valid_mse = mean_squared_error(y_valid[:, r], valid_exp)
print(f"Expectation MSE vs true ±1: train {train_mse:.4f} - val {valid_mse:.4f}")

uniform_input = np.ones((1, L))
uniform_proba = logreg.predict_proba(uniform_input)[0, 1]
uniform_exp = 2 * uniform_proba - 1
print(f"Predicted expectation at site {r} for all +1 input: {uniform_exp}")

param_count = logreg.coef_.size + logreg.intercept_.size
print(f"Number of parameters: {param_count}")


Logistic regression (site 100) train logloss: 0.3680 - val logloss: 0.3624
Expectation MSE vs true ±1: train 0.4590 - val 0.4506
Predicted expectation at site 100 for all +1 input: 0.9949232243921835
Number of parameters: 201


In [20]:
# Logistic regression with importance reweighting toward high-magnetization inputs
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, mean_squared_error

r = 100  # site index to predict (0-based)
np.random.seed(9)

X_train_lr = X_train.copy()
X_valid_lr = X_valid.copy()
X_train_lr[:, [0, -1]] = 1
X_valid_lr[:, [0, -1]] = 1

y_train_bin = ((y_train[:, r] + 1) / 2).astype(int)
y_valid_bin = ((y_valid[:, r] + 1) / 2).astype(int)

# Importance weights based on ancilla magnetization (favor near-uniform inputs)
alpha = 17.0  # larger -> stronger emphasis on |mean(X)| close to 1
mag_train = np.abs(X_train_lr.mean(axis=1))
mag_valid = np.abs(X_valid_lr.mean(axis=1))
weights_train = np.exp(alpha * (mag_train - 1.0))
weights_valid = np.exp(alpha * (mag_valid - 1.0))

logreg = LogisticRegression(max_iter=200, solver='lbfgs')
logreg.fit(X_train_lr, y_train_bin, sample_weight=weights_train)

train_proba = logreg.predict_proba(X_train_lr)[:, 1]
valid_proba = logreg.predict_proba(X_valid_lr)[:, 1]

train_logloss = log_loss(y_train_bin, train_proba, sample_weight=weights_train)
valid_logloss = log_loss(y_valid_bin, valid_proba, sample_weight=weights_valid)
print(f"Weighted logistic regression (site {r}) train logloss: {train_logloss:.4f} - val logloss: {valid_logloss:.4f}")

train_exp = 2 * train_proba - 1
valid_exp = 2 * valid_proba - 1
train_mse = mean_squared_error(y_train[:, r], train_exp, sample_weight=weights_train)
valid_mse = mean_squared_error(y_valid[:, r], valid_exp, sample_weight=weights_valid)
print(f"Weighted expectation MSE vs true ±1: train {train_mse:.4f} - val {valid_mse:.4f}")

uniform_input = np.ones((1, L))
uniform_proba = logreg.predict_proba(uniform_input)[0, 1]
uniform_exp = 2 * uniform_proba - 1
print(f"Predicted expectation at site {r} for all +1 input: {uniform_exp}")

param_count = logreg.coef_.size + logreg.intercept_.size
print(f"Number of parameters: {param_count}")


Weighted logistic regression (site 100) train logloss: 0.3909 - val logloss: 0.3738
Weighted expectation MSE vs true ±1: train 0.4705 - val 0.4420
Predicted expectation at site 100 for all +1 input: 0.9466995609565949
Number of parameters: 201
